Combining circularHeatmap, sankey, and transitionProbabilityMatrixArena. 
Input: combined_matrix_final.csv (output from stitched_labels_experiment)
Output: graphs

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
import pycirclize
from pycirclize import Circos
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
import matplotlib.cm as cm
import os
import plotly.graph_objects as go
import networkx as nx
import plotly.io as pio
from collections import defaultdict
import matplotlib
# ! pip install -U kaleido

ModuleNotFoundError: No module named 'seaborn'

In [ ]:
curr_dir = os.getcwd()
base_dir = os.path.dirname(curr_dir)
arena_data_dir = os.path.join(base_dir, '3D_Data')
arena_visualizations_dir = os.path.join( base_dir , '3D_Visualizations')

In [ ]:
# CHANGE HERE: Path to your CSV file
all_data = pd.read_csv( os.path.join( arena_data_dir , 'combined_matrix_final_total136_lc.csv') , index_col=0 )

Beginning circularHeatmap

In [ ]:
# CHANGE HERE: Path to your eps and png outputs
paths_pie_charts = [
    os.path.join(arena_visualizations_dir, 'Control/pie_charts/week1toweek3/eps/change_in_cluster_usage.eps'),
    os.path.join(arena_visualizations_dir, 'Control/pie_charts/week3toweek6/eps/change_in_cluster_usage.eps'),
    os.path.join(arena_visualizations_dir, 'Control/pie_charts/week1toweek6/eps/change_in_cluster_usage.eps')
]

# CHANGE HERE: Path to your eps and png outputs
paths_pie_charts_sorted = [
    os.path.join(arena_visualizations_dir, 'Control/pie_charts_sorted/week1toweek3/eps/change_in_cluster_usage.eps'),
    os.path.join(arena_visualizations_dir, 'Control/pie_charts_sorted/week3toweek6/eps/change_in_cluster_usage.eps'),
    os.path.join(arena_visualizations_dir, 'Control/pie_charts_sorted/week1toweek6/eps/change_in_cluster_usage.eps')
]

# CHANGE HERE: Update title variable to match what title you want
title_pie_chart = "Control"

# CHANGE HERE: Output paths for graphs AND label variables
week_1_sankey = os.path.join( arena_visualizations_dir , 'Control\sankey\week1\sankey')
week_3_sankey = os.path.join( arena_visualizations_dir , 'Control\sankey\week3\sankey')
week_6_sankey = os.path.join( arena_visualizations_dir , 'Control\sankey\week6\sankey')
all_week_sankey = os.path.join( arena_visualizations_dir , r'Control\sankey\all\sankey')

# CHANGE HERE: Path to your CSV files
grouped_clusters = pd.read_csv(os.path.join(arena_data_dir, 'groupedClusters_lc.csv'), index_col=0)

In [ ]:
week1_df = all_data[(all_data['ExperimentType'] == 'week1')]
week3_df = all_data[(all_data['ExperimentType'] == 'week3')]
week6_df = all_data[(all_data['ExperimentType'] == 'week6')]

In [ ]:
# Find clusters that are NOT in week 1
print("Clusters not present in Week 1:")
print(sorted(set(all_data['ClusterNumber'].unique().astype(int)) - set(week1_df['ClusterNumber'].unique().astype(int))))
print()

# Find clusters that are NOT in week 3
print("Clusters not present in Week 3:")
print(sorted(set(all_data['ClusterNumber'].unique().astype(int)) - set(week3_df['ClusterNumber'].unique().astype(int))))
print()

# Find clusters that are NOT in week 6
print("Clusters not present in Week 6:")
print(sorted(set(all_data['ClusterNumber'].unique().astype(int)) - set(week6_df['ClusterNumber'].unique().astype(int))))
print()

In [ ]:
week1_df = week1_df.sort_values( by='ClusterNumber' , ascending=True )
week3_df = week3_df.sort_values( by='ClusterNumber' , ascending=True )
week6_df = week6_df.sort_values( by='ClusterNumber' , ascending=True )

In [ ]:
def create_cluster_usage_df( df ):
    df = pd.DataFrame( df )

    df_cluster_usage = df['ClusterNumber'].value_counts().reset_index()
    df_cluster_usage.columns = ['ClusterNumber', 'count']
    df_cluster_usage['count'] = df_cluster_usage['count'].astype( int )
    total_count = df_cluster_usage['count'].sum()
    df_cluster_usage['percentage'] = df_cluster_usage['count'] / total_count * 100

    return df_cluster_usage

In [ ]:
week1_cluster_usage = create_cluster_usage_df( week1_df )
week3_cluster_usage = create_cluster_usage_df( week3_df )
week6_cluster_usage = create_cluster_usage_df( week6_df )

In [ ]:
# Week 1 to Week 3
def create_change_in__cluster_usage_week1toweek3( week1_cluster_usage , week3_cluster_usage ):
    merged_df = pd.merge( week1_cluster_usage , week3_cluster_usage , on='ClusterNumber' , how='right' , suffixes=('_week1' , '_week3'))
    merged_df['percentage_week1'] = merged_df['percentage_week1'].fillna(0)
    merged_df['change_in_usage'] = merged_df['percentage_week3'] - merged_df['percentage_week1']
    cluster_usage_change_week1toweek3= merged_df[['ClusterNumber', 'percentage_week1' , 'percentage_week3', 'change_in_usage']]
    return cluster_usage_change_week1toweek3

In [ ]:
cluster_usage_change_week1toweek3= create_change_in__cluster_usage_week1toweek3( week1_cluster_usage , week3_cluster_usage )
cluster_usage_change_week1toweek3 = cluster_usage_change_week1toweek3.sort_values(by='ClusterNumber')

In [ ]:
# Week 3 to Week 6
def create_change_in__cluster_usage_week3toweek6( week3_cluster_usage , week6_cluster_usage ):
    merged_df = pd.merge( week3_cluster_usage , week6_cluster_usage , on='ClusterNumber' , how='right' , suffixes=('_week3' , '_week6'))
    merged_df['percentage_week3'] = merged_df['percentage_week3'].fillna(0)
    merged_df['change_in_usage'] = merged_df['percentage_week6'] - merged_df['percentage_week3']
    cluster_usage_change_week3toweek6= merged_df[['ClusterNumber', 'percentage_week3' , 'percentage_week6', 'change_in_usage']]
    return cluster_usage_change_week3toweek6

In [ ]:
cluster_usage_change_week3toweek6= create_change_in__cluster_usage_week3toweek6( week3_cluster_usage , week6_cluster_usage )
cluster_usage_change_week3toweek6 = cluster_usage_change_week3toweek6.sort_values(by='ClusterNumber')

In [ ]:
# Week 1 to Week 6
def create_change_in__cluster_usage_week1toweek6( week1_cluster_usage , week6_cluster_usage ):
    merged_df = pd.merge( week1_cluster_usage , week6_cluster_usage , on='ClusterNumber' , how='right' , suffixes=('_week1' , '_week6'))
    merged_df['percentage_week1'] = merged_df['percentage_week1'].fillna(0)
    merged_df['change_in_usage'] = merged_df['percentage_week6'] - merged_df['percentage_week1']
    cluster_usage_change_week1toweek6= merged_df[['ClusterNumber', 'percentage_week1' , 'percentage_week6', 'change_in_usage']]
    return cluster_usage_change_week1toweek6

In [ ]:
cluster_usage_change_week1toweek6= create_change_in__cluster_usage_week1toweek6( week1_cluster_usage , week6_cluster_usage )
cluster_usage_change_week1toweek6 = cluster_usage_change_week1toweek6.sort_values(by='ClusterNumber')

In [ ]:
# Graphs ALL transitions
def create_heatmaps_three_periods(df_week1to3, df_week3to6, df_week1to6, title, save=False, paths=None):
    # CHANGE HERE: Restrict ClusterNumber values based on what clusters appear across all 3 experiment types (week1, week3, week6)
    dfs = [df[(df['ClusterNumber'] >= 1) & (df['ClusterNumber'] <= 72)] for df in [df_week1to3, df_week3to6, df_week1to6]]

    all_changes = np.concatenate([df['change_in_usage'].values for df in dfs]) # Get all change_in_usage values combined across all dfs

    # CHANGE HERE: Uncomment and comment based on which option you want
    # Option 1: Use color bar values based only on its own data. 
    # global_min = all_changes.min()
    # global_max = all_changes.max()

    # Option 2: Use color bar values based on another mouse. Ensures same scaling across different mice
    global_min = -4.774236665
    global_max = 4.531974437158014
    
    norm = TwoSlopeNorm(vmin=global_min, vcenter=0, vmax=global_max)

    titles = [
        f"Change in Cluster Usage Percentage ({title} Week 1 to Week 3)",
        f"Change in Cluster Usage Percentage ({title} Week 3 to Week 6)",
        f"Change in Cluster Usage Percentage ({title} Week 1 to Week 6)"
    ]

    for i, filtered_df in enumerate(dfs):
        clusters = filtered_df['ClusterNumber'].astype(int).values
        change_values = filtered_df['change_in_usage'].values
    
        colors = plt.cm.bwr(norm(change_values))
    
        fig, ax = plt.subplots(figsize=(6,6), subplot_kw={'projection': 'polar'})
        theta = np.linspace(0, 2 * np.pi, len(clusters), endpoint=False)
        
        slice_width = 2 * np.pi / len(clusters)
        bar_width = slice_width * 0.9
        bars = ax.bar(theta, np.ones(len(clusters)), color=colors, width=bar_width, bottom=0.5)
    
        label_offset = 0
        theta_labels = (theta + label_offset) % (2 * np.pi)
    
        ax.set_xticks(theta_labels)
        ax.set_xticklabels(clusters, fontsize=5.5, rotation=45)

        ax.set_yticklabels([])
        ax.set_yticks([])
        ax.tick_params(axis='x', pad=-8)
        ax.grid(False)
        ax.spines['polar'].set_visible(False)

        sm = plt.cm.ScalarMappable(cmap='bwr', norm=norm)
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax, orientation='horizontal', pad=0.1)
        cbar.set_label("Change in Usage Percentage")

        plt.title(titles[i])
        plt.tight_layout()

        if save:
            path = paths[i]
            eps_dir = os.path.dirname(path)
            if eps_dir:
                os.makedirs(eps_dir, exist_ok=True)
            base_filename = os.path.splitext(os.path.basename(path))[0]
            eps_path = os.path.join(eps_dir, base_filename + '.eps')
            plt.savefig(eps_path, format='eps', bbox_inches='tight')
            print(f"EPS saved to: {eps_path}")

            parent_dir = os.path.dirname(eps_dir)
            png_dir = os.path.join(parent_dir, 'png')
            os.makedirs(png_dir, exist_ok=True)
            png_path = os.path.join(png_dir, base_filename + '.png')
            plt.savefig(png_path, format='png', dpi=300, bbox_inches='tight')
            print(f"PNG saved to: {png_path}")

        plt.show()


In [ ]:
# CHANGE HERE: Update title variable to match what title you want
create_heatmaps_three_periods(
    cluster_usage_change_week1toweek3,
    cluster_usage_change_week3toweek6,
    cluster_usage_change_week1toweek6,
    title=title_pie_chart,
    save=False,
    paths=paths_pie_charts
)

In [ ]:
# Graphs ALL transitions - Sorted
def create_heatmaps_three_periods(df_week1to3, df_week3to6, df_week1to6, title, save=False, paths=None):
    # CHANGE HERE: Restrict ClusterNumber values based on what clusters appear across all 3 experiment types (week1, week3, week6)
    dfs = [df[(df['ClusterNumber'] >= 1) & (df['ClusterNumber'] <= 72)] for df in [df_week1to3, df_week3to6, df_week1to6]]
    
    all_changes = np.concatenate([df['change_in_usage'].values for df in dfs]) # Get all change_in_usage values combined across all dfs

    # CHANGE HERE: Uncomment and comment based on which option you want
    # Option 1: Use color bar values based only on its own data. 
    # global_min = all_changes.min()
    # global_max = all_changes.max()

    # Option 2: Use color bar values based on another mouse. Ensures same scaling across different mice
    global_min = -4.774236665
    global_max = 4.531974437158014

    norm = TwoSlopeNorm(vmin=global_min, vcenter=0, vmax=global_max)

    # Adjust titles based on whether you are comparing weeks or arenas - add this to input up top
    titles = [
        f"Change in Cluster Usage Percentage ({title} Week 1 to Week 3)",
        f"Change in Cluster Usage Percentage ({title} Week 3 to Week 6)",
        f"Change in Cluster Usage Percentage ({title} Week 1 to Week 6)"
    ]

    for i, filtered_df in enumerate(dfs):
        filtered_df = filtered_df.sort_values( by='change_in_usage' , ascending=True )
        clusters = filtered_df['ClusterNumber'].astype(int).values
        change_values = filtered_df['change_in_usage'].values
    
        colors = plt.cm.bwr(norm(change_values))
    
        fig, ax = plt.subplots(figsize=(6,6), subplot_kw={'projection': 'polar'})
        theta = np.linspace(0, 2 * np.pi, len(clusters), endpoint=False)
        
        slice_width = 2 * np.pi / len(clusters)
        bar_width = slice_width * 0.9
        bars = ax.bar(theta, np.ones(len(clusters)), color=colors, width=bar_width, bottom=0.5)
    
        label_offset = 0
        theta_labels = (theta + label_offset) % (2 * np.pi)
    
        ax.set_xticks(theta_labels)
        ax.set_xticklabels(clusters, fontsize=5.5, rotation=45)

        ax.set_yticklabels([])
        ax.set_yticks([])
        ax.tick_params(axis='x', pad=-8)
        ax.grid(False)
        ax.spines['polar'].set_visible(False)

        sm = plt.cm.ScalarMappable(cmap='bwr', norm=norm)
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax, orientation='horizontal', pad=0.1)
        cbar.set_label("Change in Usage Percentage")

        plt.title(titles[i])
        plt.tight_layout()

        if save:
            path = paths[i]
            eps_dir = os.path.dirname(path)
            if eps_dir:
                os.makedirs(eps_dir, exist_ok=True)
            base_filename = os.path.splitext(os.path.basename(path))[0]
            eps_path = os.path.join(eps_dir, base_filename + '.eps')
            plt.savefig(eps_path, format='eps', bbox_inches='tight')
            print(f"EPS saved to: {eps_path}")

            parent_dir = os.path.dirname(eps_dir)
            png_dir = os.path.join(parent_dir, 'png')
            os.makedirs(png_dir, exist_ok=True)
            png_path = os.path.join(png_dir, base_filename + '.png')
            plt.savefig(png_path, format='png', dpi=300, bbox_inches='tight')
            print(f"PNG saved to: {png_path}")

        plt.show()


In [ ]:
create_heatmaps_three_periods(
    cluster_usage_change_week1toweek3,
    cluster_usage_change_week3toweek6,
    cluster_usage_change_week1toweek6,
    title=title_pie_chart,
    save=False,
    paths=paths_pie_charts_sorted
)

In [ ]:
# # CHANGE HERE: Path to your CSV files
# save_path = os.path.join( arena_visualizations_dir , 'Control\pie_charts\data\cluster_usage_change_week1toweek6.csv')
# cluster_usage_change_week1toweek6.to_csv(save_path, index=False)

# save_path = os.path.join( arena_visualizations_dir , 'Control\pie_charts\data\cluster_usage_change_week3toweek6.csv')
# cluster_usage_change_week3toweek6.to_csv(save_path, index=False)

# save_path = os.path.join( arena_visualizations_dir , 'Control\pie_charts\data\cluster_usage_change_week1toweek3.csv')
# cluster_usage_change_week1toweek3.to_csv(save_path, index=False)

sankey graphs

In [ ]:
def create_sankey_plot(data, label, save=False, path=None):
    transitions = data['ClusterNumber'].shift(-1)
    transition_counts = data.groupby(['ClusterNumber', transitions]).size().unstack(fill_value=0)
    transition_probabilities = transition_counts.div(transition_counts.sum(axis=1), axis=0)

    clusters = sorted(data['ClusterNumber'].dropna().unique())
    source, target, values, colors = [], [], [], []

    # Define color map for clusters
    custom_colors = ["#FAEBD7", "#4682B4", "#2F4F4F", "#FF4500", "#8B4513", "#6A5ACD",
                     "#7B68EE", "#B22222", "#FFD700", "#CD5C5C", "#483D8B", "#A52A2A",
                     "#556B2F", "#8B008B", "#9400D3", "#FF69B4", "#DC143C", "#4B0082",
                     "#FF8C00", "#D2691E", "#FF7F50", "#228B22", "#00BFFF", "#B8860B",
                     "#6B8E23", "#C71585", "#8B0000", "#DDA0DD", "#DAA520", "#9932CC",
                     "#5F9EA0", "#008080", "#9ACD32", "#D8BFD8", "#008B8B", "#F4A460",
                     "#708090", "#D2B48C", "#778899", "#FF1493", "#7FFF00", "#00CED1",
                     "#ADFF2F", "#20B2AA", "#F5DEB3", "#E9967A", "#BA55D3", "#FF00FF",
                     "#BC8F8F", "#C0272D", "#FFE4C4", "#F5F5DC", "#DEB887", "#F0E68C",
                     "#800000", "#FF6347", "#FF4500", "#A52A2A", "#00FA9A", "#8A2BE2",
                     "#B0C4DE", "#FA8072", "#87CEEB", "#708090", "#40E0D0", "#6B8E23",
                     "#48D1CC", "#00FF7F", "#4682B4", "#AFEEEE", "#DAA520", "#7FFFD4"]
    color_map = {f"Cluster {int(i)}": custom_colors[int(i) % len(custom_colors)] for i in clusters}

    for col in transition_probabilities.columns:
        for row in transition_probabilities.index:
            prob = transition_probabilities.at[row, col]
            if prob > 0:
                src = f"Cluster {int(row)}"
                tgt = f"Cluster {int(col)}"
                source.append(src)
                target.append(tgt)
                values.append(prob)
                colors.append(color_map[src])

    def unique_ordered(seq):
        seen = set()
        return [x for x in seq if not (x in seen or seen.add(x))]

    cluster_labels = unique_ordered(source + target)
    source_labels = [f"{label} (from)" for label in cluster_labels]
    target_labels = [f"{label} (to)" for label in cluster_labels]
    all_labels = source_labels + target_labels
    label_indices = {f"{label} (from)": i for i, label in enumerate(cluster_labels)}
    label_indices.update({f"{label} (to)": i + len(cluster_labels) for i, label in enumerate(cluster_labels)})

    fig = go.Figure(go.Sankey(
        node=dict(
            pad=20, thickness=25, line=dict(color="black", width=0.5),
            label=all_labels, color="lightgray"),
        link=dict(
            source=[label_indices[f"{s} (from)"] for s in source],
            target=[label_indices[f"{t} (to)"] for t in target],
            value=values, color=colors)
    ))

    fig.update_layout(
        title_text=f"Cluster Transitions in {label}",
        font_size=12, height=1000, width=1000
    )

    if save and path:
        base = os.path.splitext(path)[0]
        svg_path = base + ".svg"
        png_path = base + ".png"
        os.makedirs(os.path.dirname(svg_path), exist_ok=True)
        fig.write_image(svg_path)
        fig.write_image(png_path)
        print(f"SVG saved to: {svg_path}")
        print(f"PNG saved to: {png_path}")

    fig.show()

In [ ]:
# Adjust labels based on whether you are comparing weeks or arenas - add this to input up top
create_sankey_plot(all_data[all_data["ExperimentType"] == 'week1'], label="Week 1" , save = False , path = week_1_sankey)
create_sankey_plot(all_data[all_data["ExperimentType"] == 'week3'], label="Week 3" , save = False , path = week_3_sankey)
create_sankey_plot(all_data[all_data["ExperimentType"] == 'week6'], label="Week 6" , save = False , path = week_6_sankey)
create_sankey_plot(all_data, label="All Weeks Combined" , save = False , path = all_week_sankey)

In [ ]:
custom_colors = [
    "#e6194b", "#3cb44b", "#ffe119", "#4363d8", "#f58231", "#911eb4", "#46f0f0", "#f032e6",
    "#bcf60c", "#fabebe", "#008080", "#e6beff", "#9a6324", "#fffac8", "#800000", "#aaffc3",
    "#808000", "#ffd8b1", "#000075", "#80F080", "#00FFF5", "#000000", "#ff4500", "#32cd32",
]

group_list = sorted(grouped_clusters['Group'].dropna().unique())
color_map = {int(group): custom_colors[i % len(custom_colors)] for i, group in enumerate(group_list)}

def map_grouped_clusters(data):
    return data.merge(grouped_clusters[['Cluster', 'Group']], left_on='ClusterNumber', right_on='Cluster', how='left')

def create_grouped_sankey_plot(data, label, save=False, path=None):
    data = map_grouped_clusters(data)
    data['Next_Group'] = data['Group'].shift(-1)
    data = data.dropna(subset=['Group', 'Next_Group'])
    data['Group'] = data['Group'].astype(int)
    data['Next_Group'] = data['Next_Group'].astype(int)

    unique_groups = sorted(pd.concat([data['Group'], data['Next_Group']]).unique())
    group_labels = [f"Group {g}" for g in unique_groups]
    n = len(unique_groups)
    label_indices = {f"Group {g}": i for i, g in enumerate(unique_groups)}

    transition_counts = data.groupby(['Group', 'Next_Group']).size().unstack(fill_value=0)
    transition_probabilities = transition_counts.div(transition_counts.sum(axis=1), axis=0)

    source, target, values, colors = [], [], [], []

    for col in transition_probabilities.columns:
        for row in transition_probabilities.index:
            prob = transition_probabilities.at[row, col]
            if prob > 0:
                src = f"Group {int(row)}"
                tgt = f"Group {int(col)}"
                source.append(label_indices[src])
                target.append(label_indices[tgt] + n)
                values.append(prob)
                colors.append(color_map[int(row)])

    y_spacing = 1.0 / (n + 1)
    y_positions = [y_spacing * (i + 1) for i in range(n)] * 2
    x_positions = [0.1] * n + [0.9] * n

    fig = go.Figure(go.Sankey(
        node=dict(
            pad=20,
            thickness=25,
            line=dict(color="black", width=0.5),
            label=group_labels * 2,
            color="lightgray",
            x=x_positions,
            y=y_positions
        ),
        link=dict(
            source=source,
            target=target,
            value=values,
            color=colors
        )
    ))

    fig.update_layout(
        title_text=f"Grouped Cluster Transitions in {label}",
        font_size=12,
        height=1000,
        width=1000
    )

    if save and path:
        base = os.path.splitext(path)[0]
        svg_path = base + ".svg"
        png_path = base + ".png"
        os.makedirs(os.path.dirname(svg_path), exist_ok=True)
        fig.write_image(svg_path)
        fig.write_image(png_path)
        print(f"SVG saved to: {svg_path}")
        print(f"PNG saved to: {png_path}")

    fig.show()

In [ ]:
all_data['ExperimentType'] = all_data['ExperimentType'].str.lower()

# CHANGE HERE: Output path for graphs AND label variables - MOVE THIS UP TOP
create_grouped_sankey_plot(
    all_data[all_data['ExperimentType'] == 'week1'],
    label="Week 1",
    save=False,
    path=os.path.join(arena_visualizations_dir, 'Control/sankey_grouped/week1/sankey')
)
create_grouped_sankey_plot(
    all_data[all_data['ExperimentType'] == 'week3'],
    label="Week 3",
    save=False,
    path=os.path.join(arena_visualizations_dir, 'Control/sankey_grouped/week3/sankey')
)
create_grouped_sankey_plot(
    all_data[all_data['ExperimentType'] == 'week6'],
    label="Week 6",
    save=False,
    path=os.path.join(arena_visualizations_dir, 'Control/sankey_grouped/week6/sankey')
)
create_grouped_sankey_plot(
    all_data,
    label="All Weeks Combined",
    save=False,
    path=os.path.join(arena_visualizations_dir, 'Control/sankey_grouped/all/sankey')
)

transitionProbabilityMatrixArena graphs